In [1]:
import torch
import torchvision.transforms as transforms
import logging
import importlib
import numpy as np

from components import FL_sim
from components.other_utilities import models_to_train
import components.broadcast_components.reporting_utilities
import components.broadcast_components.broadcasting_process.WZ_broadcast
import components.other_utilities.datasets

importlib.reload(FL_sim)
importlib.reload(models_to_train)
importlib.reload(components.broadcast_components.reporting_utilities)
importlib.reload(components.other_utilities.datasets)
importlib.reload(components.broadcast_components.broadcasting_process.WZ_broadcast)

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from components.broadcast_components.broadcasting_process.WZ_broadcast import WZBroadcastProtocol

torch.set_float32_matmul_precision('high')

In [2]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]
#             )
#         ])
#     ) for s in ['train', 'val']]


dataset = [
    FasterSVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

# dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

In [3]:
importlib.reload(components.broadcast_components.reporting_utilities)
from components.broadcast_components.reporting_utilities import ReportingUtilities

reporting_util = ReportingUtilities()
broadcast_prot = WZBroadcastProtocol(4,'RNN', train_sample_size=100_000,
                                    metric_report_flag=True, code_bit_size=2, lr=1e-5)
@reporting_util.record_compr_stats
def pre_send_process(worker_grad_dict, agent_id):
    worker_broadcast_data = broadcast_prot.wz_encoding_process(worker_grad_dict, agent_id)
    return worker_broadcast_data


@reporting_util.report_reconst_wrapper
def server_rec_process(worker_broadcast_data, agent_id, worker_count, global_model_dims, previous_data, ):
    result_dict = broadcast_prot.wz_reconstruction_process(
        worker_broadcast_data, agent_id, worker_count, global_model_dims, previous_data, )
    return result_dict


def applied_on_grads_before_optim(fl_model, worker_id, curr_round, current_epoch, batch_idx, *args, **kwargs):
    # save_grads_f_applied_on_grads(fl_model,
    #     worker_id, curr_round, current_epoch, batch_idx, *args, **kwargs)
    pass

In [ ]:
# def pre_send_process(worker_grad_dict, agent_id):
#     return [worker_grad_dict, np.array([0]*len(worker_grad_dict)), np.array([1]*len(worker_grad_dict)), int]
#
# def server_rec_process(worker_broadcast_data, agent_id, worker_count, global_model_dims, previous_data, ):
#     return worker_broadcast_data[0]

In [ ]:
k = 5  # Number of workers
logging_disabled=True
post_training_report=True

k = 3
logging_disabled=True
post_training_report=False

batch_size = 5000
# batch_size /= 6 # more batches as samples
# batch_size /= 50 # imagenet
# batch_size /= 3 # resnet 50
batch_size = int(batch_size)

# model.args_for_f_on_grad['save_folder_path'] = \
#     f"experiments/exp_data/gradients_resnet/gradients_resnet_t{i}/"

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.01,
            logging_disabled=logging_disabled, applied_on_grads_before_optim=applied_on_grads_before_optim)

# model.load_state_dict(torch.load('experiments/exp_data/resnet18_svhn.pth', map_location='cpu'))

sim = FLSimulator(
    num_agents=k, communication_rounds=10, client_epochs_per_round=15,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, pl_model=model,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

# pre-training global model before saving it
# sim.do_train_global_model_and_set_local_model(num_epochs=4)
# torch.save(sim.global_model.state_dict(), 'experiments/exp_data/resnet18_svhn.pth')

import warnings
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="Starting from v1.9.0, `tensorboardX` has been removed")
warnings.filterwarnings("ignore", message="The 'val_dataloader' does not have many workers which may be a bottleneck")

sim.run_simulation(post_training_report=post_training_report, pre_training_global_epochs=3)